
# 第四章 数学公式与函数可视化

- 4.2 符号分类（k-NN、SVM）
- 4.3 马尔可夫链旋律生成
- 4.4 RNN 与 LSTM
- 4.5 约束生成

**输出目录**：`output_figures/`

**依赖包**：numpy, matplotlib, scikit-learn


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Circle, Rectangle
import matplotlib.patches as mpatches

# 从当前目录向上定位项目根
_p = os.getcwd()
while not os.path.exists(os.path.join(_p, 'CODE', 'datasets')):
    _parent = os.path.dirname(_p)
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/datasets 的目录），请在项目内运行本 Notebook")
    _p = _parent
FIGURES_DIR = os.path.join(_p, 'CODE', 'chapter04', 'output_figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 600,
    'figure.facecolor': 'white',
    'savefig.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': 'black',
    'axes.linewidth': 0.8,
    'grid.color': '#cccccc',
    'grid.linewidth': 0.5,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
plt.rcParams['font.sans-serif'] = [
    'PingFang SC', 'Hiragino Sans GB', 'Microsoft YaHei',
    'SimHei', 'Noto Sans CJK SC', 'DejaVu Sans'
]
plt.rcParams['axes.unicode_minus'] = False

print("环境配置完成。输出目录:", FIGURES_DIR)



## 图 1：欧氏距离的几何解释

对应公式（eq:knn-distance）：

$$d(\mathbf{x}, \mathbf{x}_i) = \|\mathbf{x} - \mathbf{x}_i\|_2 = \sqrt{\sum_{j=1}^d (x_j - x_{i,j})^2}$$

二维示意：两个特征维度上的点，连线长度即欧氏距离。


In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))

p1 = np.array([1, 2])
p2 = np.array([4, 5])

ax.scatter(*p1, color='#2c3e50', s=120, zorder=5)
ax.scatter(*p2, color='#7f8c8d', s=120, zorder=5)
ax.plot([p1[0], p2[0]], [p1[1], p2[1]], 'k--', linewidth=1.2, zorder=3)
ax.plot([p1[0], p2[0]], [p1[1], p1[1]], 'k:', linewidth=0.8, alpha=0.6)
ax.plot([p2[0], p2[0]], [p1[1], p2[1]], 'k:', linewidth=0.8, alpha=0.6)

ax.annotate(r'$\mathbf{x}_i = (x_{i,1}, x_{i,2})$', xy=p1, xytext=(p1[0]-1.2, p1[1]-0.4), fontsize=11)
ax.annotate(r'$\mathbf{x} = (x_1, x_2)$', xy=p2, xytext=(p2[0]+0.2, p2[1]+0.2), fontsize=11)
ax.annotate(r'$\Delta x_1$', xy=(2.5, 2.15), fontsize=10)
ax.annotate(r'$\Delta x_2$', xy=(4.15, 3.5), fontsize=10)

mid = (p1 + p2) / 2
ax.annotate(r'$d = \sqrt{(\Delta x_1)^2 + (\Delta x_2)^2}$',
            xy=mid, xytext=(mid[0]-2.5, mid[1]+0.6),
            fontsize=11, color='#c0392b',
            arrowprops=dict(arrowstyle='->', color='#c0392b', lw=0.8))

ax.set_xlim(-0.5, 6)
ax.set_ylim(-0.5, 6.5)
ax.set_aspect('equal')
ax.set_xlabel('特征维度 1')
ax.set_ylabel('特征维度 2')
ax.set_title('欧氏距离')
ax.grid(True, alpha=0.3)

out_path = os.path.join(FIGURES_DIR, 'fig_euclidean_distance.png')
fig.savefig(out_path, dpi=600, bbox_inches='tight')
plt.show()
print(f"已保存: {out_path}")



## 图 2：激活函数曲线

对应公式（eq:tanh-sigmoid）：

$$\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}, \qquad \sigma(z) = \frac{1}{1 + e^{-z}}$$

直观展示：
- 非线性压缩效果
- 饱和区域（|z| 很大时输出趋近边界）
- sigmoid 的 $(0,1)$ 值域如何对应 LSTM 门控比例

**图题：激活函数 $\tanh$ 与 $\sigma$ 的曲线。**


In [ ]:
z = np.linspace(-5, 5, 500)
tanh_y = np.tanh(z)
sigmoid_y = 1 / (1 + np.exp(-z))

fig, ax = plt.subplots(figsize=(7, 4.5))

ax.plot(z, tanh_y, 'k-', linewidth=2, label=r'$\tanh(z) \in (-1, 1)$')
ax.plot(z, sigmoid_y, 'k--', linewidth=2, label=r'$\sigma(z) \in (0, 1)$')

ax.axvspan(-5, -2.5, color='gray', alpha=0.08)
ax.axvspan(2.5, 5, color='gray', alpha=0.08)

ax.annotate('饱和区域（导数趋近 0）', xy=(2, 1.11), fontsize=11, color='gray')
ax.annotate('饱和区域', xy=(-4.8, -1.15), fontsize=11, color='gray')
ax.annotate('两函数在 $z=0$ 处斜率最大', xy=(0.2, 0.1), fontsize=10, color='gray')
ax.annotate('Sigmoid 门值：接近 0 时强抑制，接近 1 时近似保留',
            xy=(2.5, 0.92), xytext=(-4.9, 0.88),
            fontsize=10, color='#c0392b',
            arrowprops=dict(arrowstyle='->', color='#c0392b', lw=0.8))

ax.axhline(0, color='black', linewidth=0.5)
ax.axhline(1, color='gray', linestyle=':', linewidth=0.8, alpha=0.7)
ax.axhline(-1, color='gray', linestyle=':', linewidth=0.8, alpha=0.7)
ax.set_xlim(-5, 5)
ax.set_ylim(-1.3, 1.3)
ax.set_xlabel(r'激活函数输入 $z$（仿射变换后的值）')
ax.set_ylabel(r'输出')
ax.legend(loc='lower right', framealpha=0.9)
ax.grid(True, alpha=0.3)

out_path = os.path.join(FIGURES_DIR, 'fig_activation_functions.png')
fig.savefig(out_path, dpi=600, bbox_inches='tight')
plt.show()
print(f"已保存: {out_path}")



## 图 3：Softmax 变换

对应公式（eq:softmax）：

$$P(y=j \mid x) = \frac{\exp(z_j)}{\sum_{k=1}^{|\mathcal{V}|} \exp(z_k)}$$

直观展示：原始分数（可正可负）→ 概率（全正，和为 1）。


In [ ]:
logits = np.array([2.5, 1.8, 1.2, 0.5, -0.3, -1.0, -2.0])
labels = [f'token {chr(65+i)}' for i in range(len(logits))]
probs = np.exp(logits) / np.sum(np.exp(logits))

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

bars1 = axes[0].bar(range(len(logits)), logits, color='#7f8c8d', edgecolor='black', linewidth=0.5)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_xticks(range(len(labels)))
axes[0].set_xticklabels(labels, rotation=30, ha='right')
axes[0].set_ylabel('logit 分数')
axes[0].set_title('原始分数 $z_j$（可正可负）')
axes[0].set_ylim(-3, 4)
axes[0].grid(axis='y', alpha=0.3)

for bar, v in zip(bars1, logits):
    axes[0].annotate(f'{v:.1f}', xy=(bar.get_x() + bar.get_width()/2, v),
                     xytext=(0, 5 if v >= 0 else -12), textcoords='offset points',
                     ha='center', fontsize=9)

bars2 = axes[1].bar(range(len(probs)), probs, color='#2c3e50', edgecolor='black', linewidth=0.5)
axes[1].set_xticks(range(len(labels)))
axes[1].set_xticklabels(labels, rotation=30, ha='right')
axes[1].set_ylabel('概率')
axes[1].set_title('Softmax 后 $P(y=j)$（> 0，和为 1）')
axes[1].set_ylim(0, 0.6)
axes[1].grid(axis='y', alpha=0.3)

for bar, v in zip(bars2, probs):
    axes[1].annotate(f'{v:.3f}', xy=(bar.get_x() + bar.get_width()/2, v),
                     xytext=(0, 5), textcoords='offset points',
                     ha='center', fontsize=8)

axes[1].annotate('sum = 1.000', xy=(0.98, 0.95), xycoords='axes fraction',
                 fontsize=10, ha='right', va='top',
                 bbox=dict(boxstyle='round', facecolor='white', edgecolor='gray', alpha=0.8))

fig.patches.append(FancyArrowPatch((0.46, 0.93), (0.54, 0.93), transform=fig.transFigure,
    arrowstyle='->', mutation_scale=25, linewidth=1.5, color='#c0392b'))
fig.text(0.5, 0.97, '取指数并归一化', ha='center', va='center', fontsize=11, color='#c0392b', transform=fig.transFigure)

plt.tight_layout()
out_path = os.path.join(FIGURES_DIR, 'fig_softmax_transform.png')
fig.savefig(out_path, dpi=600, bbox_inches='tight')
plt.show()
print(f"已保存: {out_path}")



## 图 4：温度参数对概率分布的影响

对应公式（eq:temperature-softmax）：

$$P_T(y=j) = \frac{\exp(z_j / T)}{\sum_{k=1}^{|\mathcal{V}|} \exp(z_k / T)}$$

同一个 logits，三种温度下的概率分布对比。

**图题：温度参数改变概率分布的尖锐程度。**


In [ ]:
logits = np.array([2.5, 1.8, 1.2, 0.5, -0.3, -1.0, -2.0])
labels = [f'{chr(65+i)}' for i in range(len(logits))]
temperatures = [0.5, 1.0, 2.0]
colors = ['#2c3e50', '#7f8c8d', '#bdc3c7']

fig, axes = plt.subplots(1, 3, figsize=(11, 3.8), sharey=True)

for ax, T, color in zip(axes, temperatures, colors):
    probs = np.exp(logits / T) / np.sum(np.exp(logits / T))
    bars = ax.bar(range(len(probs)), probs, color=color, edgecolor='black', linewidth=0.5)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels)
    ax.set_ylabel('概率' if T == 0.5 else '')
    ax.set_title(f'$T = {T}$')
    ax.set_ylim(0, 0.8)
    ax.grid(axis='y', alpha=0.3)
    
    entropy = -np.sum(probs * np.log2(probs + 1e-12))
    ax.annotate(f'熵 = {entropy:.2f} bits', xy=(0.5, 0.92), xycoords='axes fraction',
                ha='center', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='white', edgecolor='gray', alpha=0.7))
    
    ax.annotate(f'max = {probs.max():.3f}', xy=(np.argmax(probs), probs.max()),
                xytext=(6, 4), textcoords='offset points', ha='center', fontsize=9)

fig.text(0.5, -0.01, 'T<1: 概率更集中，T=1: 原 softmax，T>1: 概率更平坦',
         ha='center', fontsize=10, transform=fig.transFigure)

plt.tight_layout()
out_path = os.path.join(FIGURES_DIR, 'fig_temperature_probability.png')
fig.savefig(out_path, dpi=600, bbox_inches='tight')
plt.show()
print(f"已保存: {out_path}")



## 图 5：交叉熵损失

对应公式（eq:cross-entropy-next-token）：

$$\mathcal{L} = -\log P(y_{\mathrm{true}} \mid x)$$

当模型对真实 token 给出的概率越高，损失越小；概率越低，损失越大（趋向 +∞）。

**图题：交叉熵损失与模型对真实 token 概率的关系。**


In [ ]:
p = np.linspace(0.001, 1.0, 500)
loss = -np.log(p)

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(p, loss, 'k-', linewidth=2)

key_probs = [0.1, 0.5, 0.9, 0.99]
offsets = [(6, 10), (10, 7), (7, 12), (15.5, 10)]
has = ['left', 'left', 'right', 'right']
for kp, (dx, dy), ha in zip(key_probs, offsets, has):
    kl = -np.log(kp)
    ax.plot(kp, kl, 'ko', markersize=6)
    ax.annotate(f'P={kp:.2f}\nL={kl:.2f}', xy=(kp, kl),
                xytext=(dx, dy), textcoords='offset points',
                fontsize=9, ha=ha,
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='gray', alpha=0.8))

ax.axhline(0, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
ax.set_xlim(0, 1.05)
ax.set_ylim(-0.5, 8)
ax.set_xlabel(r'模型对真实 token 的概率 $P(y_{true} | x)$')
ax.set_ylabel(r'交叉熵损失 $L = -\log P$')
ax.grid(True, alpha=0.3)

ax.axvspan(0, 0.2, color='#e74c3c', alpha=0.1)
ax.text(0.21, 6.5, '真实 token 概率较低区域', ha='center', fontsize=9, color='#c0392b')

out_path = os.path.join(FIGURES_DIR, 'fig_cross_entropy_loss.png')
fig.savefig(out_path, dpi=600, bbox_inches='tight')
plt.show()
print(f"已保存: {out_path}")



## 图 6：约束采样——掩码与重新归一化

对应公式（eq:pentatonic-mask 与 eq:masked-renormalization）：

$$m_j = \begin{cases} 1, & \text{合法}\\ 0, & \text{非法} \end{cases}, \qquad \tilde{p}_j = \frac{m_j p_j}{\sum_i m_i p_i}$$

展示：原始概率 → 施加五声音阶掩码 → 非法候选归零 → 合法候选重新归一化。

**图题：约束采样：在合法集合内重新分配概率。**


In [ ]:
tokens = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B', '休止']
logits = np.array([2.0, 0.5, 1.8, 0.3, 1.5, 0.2, 0.4, 2.2, 0.1, 1.6, 0.2, 0.3, 1.0])
probs = np.exp(logits) / np.sum(np.exp(logits))

mask = np.array([1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1])
masked = probs * mask
constrained = masked / masked.sum()

fig, axes = plt.subplots(1, 3, figsize=(12, 4.4))

bars1 = axes[0].bar(range(len(tokens)), probs, color='#7f8c8d', edgecolor='black', linewidth=0.5)
axes[0].set_xticks(range(len(tokens)))
axes[0].set_xticklabels(tokens, rotation=45, ha='right')
axes[0].set_ylabel('概率')
axes[0].set_title('原始概率 $p_j$')
axes[0].set_ylim(0, 0.3)
axes[0].grid(axis='y', alpha=0.3)

bars2 = axes[1].bar(range(len(tokens)), masked, color='#bdc3c7', edgecolor='black', linewidth=0.5)
for i, (bar, m) in enumerate(zip(bars2, mask)):
    if m == 0:
        bar.set_color('#ecf0f1')
        bar.set_edgecolor('#bdc3c7')
        bar.set_linewidth(0.5)
        bar.set_linestyle('--')
axes[1].set_xticks(range(len(tokens)))
axes[1].set_xticklabels(tokens, rotation=45, ha='right')
axes[1].set_title('施加掩码 $m_j$（不合法候选置零）')
axes[1].set_ylim(0, 0.3)
axes[1].grid(axis='y', alpha=0.3)

bars3 = axes[2].bar(range(len(tokens)), constrained, color='#2c3e50', edgecolor='black', linewidth=0.5)
for i, (bar, m) in enumerate(zip(bars3, mask)):
    if m == 0:
        bar.set_color('#ecf0f1')
        bar.set_edgecolor('#bdc3c7')
axes[2].set_xticks(range(len(tokens)))
axes[2].set_xticklabels(tokens, rotation=45, ha='right')
axes[2].set_title(r'重新归一化 $\tilde{p}_j$')
axes[2].set_ylim(0, 0.3)
axes[2].grid(axis='y', alpha=0.3)

fig.patches.append(FancyArrowPatch((0.34, 0.75), (0.39, 0.75), transform=fig.transFigure,
    arrowstyle='->', mutation_scale=20, linewidth=1.5, color='#c0392b'))
fig.text(0.365, 0.79, '逐元素相乘', ha='center', va='center',
         fontsize=12, color='#c0392b', transform=fig.transFigure)

fig.patches.append(FancyArrowPatch((0.66, 0.75), (0.71, 0.75), transform=fig.transFigure,
    arrowstyle='->', mutation_scale=20, linewidth=1.5, color='#c0392b'))
fig.text(0.68, 0.79, '归一化', ha='center', va='center',
         fontsize=12, color='#c0392b', transform=fig.transFigure)

legend_elements = [
    mpatches.Patch(facecolor='#2c3e50', edgecolor='black', label='合法候选'),
    mpatches.Patch(facecolor='#ecf0f1', edgecolor='#bdc3c7', linestyle='--', label='非法候选（概率归零）'),
]
fig.legend(handles=legend_elements, loc='upper center', ncol=2, framealpha=0.9,
           bbox_to_anchor=(0.5, 0.18))

plt.tight_layout(rect=(0, 0.14, 1, 0.80))
out_path = os.path.join(FIGURES_DIR, 'fig_constrained_sampling_math.png')
fig.savefig(out_path, dpi=600, bbox_inches='tight')
plt.show()
print(f"已保存: {out_path}")



## 图 7：马尔可夫链的依赖结构

对应公式（eq:markov-kth）：

$$\mathcal{P}(y_t \mid y_{t-1}, \dots, y_{t-k}) = \frac{c(y_{t-k}, \dots, y_{t-1}, y_t)}{c(y_{t-k}, \dots, y_{t-1})}$$

展示一阶、二阶链与 RNN 所使用的历史信息范围。

**图题：不同阶数马尔可夫链与 RNN 的历史依赖方式。**


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))

def draw_markov_chain(ax, order, title, has_rnn=False):
    n_steps = 6
    y_pos = 0.5
    
    for t in range(n_steps):
        circle = Circle((t, y_pos), 0.28, facecolor='white', edgecolor='black', linewidth=1.2, zorder=5)
        ax.add_patch(circle)
        ax.text(t, y_pos, f'$y_{{{t}}}$', ha='center', va='center', fontsize=12, zorder=6)
        if not has_rnn:
            ax.text(t, y_pos - 0.75, f't={t}', ha='center', va='top', fontsize=9, color='gray')
    
    target = n_steps - 1
    if order == 1:
        ax.annotate('', xy=(target - 0.25, y_pos), xytext=(target - 1 + 0.2, y_pos),
                    arrowprops=dict(arrowstyle='->', color='#c0392b', lw=1.5))
        ax.text(target - 0.5, y_pos + 0.4, '只看 1 步', ha='center', fontsize=9, color='#c0392b')
    elif order == 2:
        for src in [target - 2, target - 1]:
            ax.annotate('', xy=(target - 0.25, y_pos + 1), xytext=(src + 0.2, y_pos),
                        arrowprops=dict(arrowstyle='->', color='#c0392b', lw=1.5))
        ax.text(target - 1, y_pos + 0.4, '看 2 步', ha='center', fontsize=9, color='#c0392b')
    
    if has_rnn:
        state_y = -0.05
        state_w, state_h = 0.46, 0.25
        for src_idx in range(target):
            rect = Rectangle((src_idx - state_w/2, state_y - state_h/2), state_w, state_h,
                             facecolor='#ecf0f1', edgecolor='#2c3e50', linewidth=0.9)
            ax.add_patch(rect)
            ax.text(src_idx, state_y, f'$s_{{{src_idx}}}$', ha='center', va='center',
                    fontsize=9, color="#243342")
            ax.annotate('', xy=(src_idx, state_y + state_h/2 -0.05),
                        xytext=(src_idx, y_pos - 0.2),
                        arrowprops=dict(arrowstyle='->', color='#7f8c8d', lw=0.6))
            if src_idx > 0:
                ax.annotate('', xy=(src_idx - state_w/2, state_y),
                            xytext=(src_idx - 1 + state_w/2, state_y),
                            arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=1.1))
        ax.annotate('', xy=(target - 0.23, y_pos - 0.13),
                    xytext=(target - 1 + state_w/2, state_y),
                    arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=1.2))
        ax.text((target - 1) / 2, state_y - 0.32, '历史经递归状态逐步传递',
                ha='center', va='top', fontsize=9, color='#2c3e50')
        ax.text(target - 0.85, y_pos + 0.5, '较早输入可经状态链影响后续',
                ha='center', fontsize=9, color='#2c3e50')
    
    ax.set_xlim(-0.6, n_steps - 0.4)
    ax.set_ylim(-0.55, 1.2)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=12, pad=10)

draw_markov_chain(axes[0], 1, '一阶马尔可夫链 ($k=1$)')
draw_markov_chain(axes[1], 2, '二阶马尔可夫链 ($k=2$)')
draw_markov_chain(axes[2], 0, 'RNN / LSTM', has_rnn=True)

plt.tight_layout()
out_path = os.path.join(FIGURES_DIR, 'fig_markov_dependency_structure.png')
fig.savefig(out_path, dpi=600, bbox_inches='tight')
plt.show()
print(f"已保存: {out_path}")



## 图 8：SVM 最大间隔与决策边界

对应公式（eq:svm-decision 与 eq:svm-primal）：

$$f(\mathbf{x}) = \mathbf{w}^\top \mathbf{x} + b, \qquad \min_{\mathbf{w}, b} \frac{1}{2}\|\mathbf{w}\|_2^2 \quad \text{s.t.} \quad y_i(\mathbf{w}^\top \mathbf{x}_i + b) \geq 1$$

几何示意：超平面、间隔边界、支持向量。下方代码使用线性软间隔 SVC（$C=1$）绘图，因此样本可以进入间隔或越过决策边界；上式的硬间隔约束只在线性可分时可行。


In [ ]:
np.random.seed(42)

n_pos, n_neg = 15, 15
mean_pos, mean_neg = [2, 2], [-1, -1]
cov = [[1.5, 0.5], [0.5, 1.5]]

X_pos = np.random.multivariate_normal(mean_pos, cov, n_pos)
X_neg = np.random.multivariate_normal(mean_neg, cov, n_neg)
X = np.vstack([X_pos, X_neg])
y = np.array([1] * n_pos + [-1] * n_neg)

from sklearn import svm
clf = svm.SVC(kernel='linear', C=1.0)
clf.fit(X, y)

w, b = clf.coef_[0], clf.intercept_[0]
support_vectors = clf.support_vectors_

xx = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 200)
yy = np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 200)
XX, YY = np.meshgrid(xx, yy)
Z = w[0] * XX + w[1] * YY + b

fig, ax = plt.subplots(figsize=(6.5, 5.5))

ax.contour(XX, YY, Z, levels=[-1, 0, 1], colors='black', linestyles=['--', '-', '--'], linewidths=[1, 1.5, 1])
ax.contourf(XX, YY, Z, levels=[-100, -1, 1, 100], colors=['#ecf0f1', '#fdfefe', '#ecf0f1'], alpha=0.3)

ax.scatter(X_pos[:, 0], X_pos[:, 1], c='#2c3e50', s=80, marker='o', edgecolors='black', linewidth=0.5, label='正类', zorder=5)
ax.scatter(X_neg[:, 0], X_neg[:, 1], c='#7f8c8d', s=80, marker='s', edgecolors='black', linewidth=0.5, label='负类', zorder=5)
ax.scatter(support_vectors[:, 0], support_vectors[:, 1], s=200, facecolors='none', edgecolors='#c0392b', linewidths=1.5, label='支持向量', zorder=6)

ax.annotate('决策边界\nf(x) = 0', xy=(4.35, 0.2), fontsize=10, ha='center',
            bbox=dict(boxstyle='round', facecolor='white', edgecolor='black', alpha=0.9))
ax.annotate('间隔边界\nf(x) = +1', xy=(4, 2.5), fontsize=9, ha='center', color='gray')
ax.annotate('间隔边界\nf(x) = -1', xy=(-2, -2.6), fontsize=9, ha='center', color='gray')

mid_x, mid_y = 2.0, -2.5
norm_w = w / np.linalg.norm(w)
point_on_boundary = np.array([mid_x, mid_y]) - (w[0]*mid_x + w[1]*mid_y + b) / np.linalg.norm(w)**2 * w
point_plus = point_on_boundary + norm_w / np.linalg.norm(w)
point_minus = point_on_boundary - norm_w / np.linalg.norm(w)
ax.annotate('', xy=point_plus, xytext=point_minus, arrowprops=dict(arrowstyle='<->', color='#c0392b', lw=1.2))
ax.text(point_on_boundary[0] - 2, point_on_boundary[1] - 1.3, r'间隔 = $2 / \|\mathbf{w}\|_2$', fontsize=10, color='#c0392b')

ax.set_xlabel('特征维度 1')
ax.set_ylabel('特征维度 2')
ax.set_title('SVM 最大间隔：支持向量参与确定边界')
ax.legend(loc='upper left', bbox_to_anchor=(1, 1), framealpha=0.9)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

out_path = os.path.join(FIGURES_DIR, 'fig_svm_boundary_math.png')
fig.savefig(out_path, dpi=600, bbox_inches='tight')
plt.show()
print(f"已保存: {out_path}")



## 图 9：RNN 隐藏状态更新

对应公式（eq:rnn-hidden）：

$$h_t = \tanh(W_{xh} e_t + W_{hh} h_{t-1} + b_h)$$

展示"当前输入 + 上一时刻隐藏状态 → 新隐藏状态"的循环结构。


In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))

steps = 4
y_center = 0.5
box_w, box_h = 0.6, 0.5

for t in range(steps):
    x = t * 2.0
    rect = Rectangle((x - box_w/2, y_center - box_h/2), box_w, box_h,
                     facecolor='#ecf0f1', edgecolor='black', linewidth=1.2)
    ax.add_patch(rect)
    ax.text(x, y_center, f'$h_{{{t}}}$', ha='center', va='center', fontsize=13, fontweight='bold')
    
    ax.text(x, y_center - 0.48, f'$e_{{{t}}}$', ha='center', va='top', fontsize=11, color='#7f8c8d')
    ax.annotate('', xy=(x, y_center - box_h/2 - 0.02), xytext=(x, y_center - 0.48),
                arrowprops=dict(arrowstyle='->', color='#7f8c8d', lw=1))
    
    if t < steps - 1:
        ax.annotate('', xy=(x + 2.0 - box_w/2 - 0.05, y_center), xytext=(x + box_w/2 + 0.05, y_center),
                    arrowprops=dict(arrowstyle='->', color='#c0392b', lw=1.5, connectionstyle='arc3,rad=0.15'))
        ax.text(x + 1.0, y_center + 0.45, r'$W_{hh}$', ha='center', fontsize=10, color='#c0392b')
    
    ax.text(x, y_center + 0.5, f'$\hat{{y}}_{{{t+1}}}$', ha='center', va='bottom', fontsize=11, color='#2c3e50')
    ax.annotate('', xy=(x, y_center + 0.5), xytext=(x, y_center + box_h/2 + 0.02),
                arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=1))

ax.text((steps - 1) / 2 * 2.0, y_center - 0.1, r'$tanh$', ha='center', va='center',
        fontsize=12, color='#7f8c8d', alpha=0.9)

eq_text = r'$h_t = tanh(W_{xh} e_t + W_{hh} h_{t-1} + b_h)$'
ax.text((steps - 1) / 2 * 2.0, -0.2, eq_text, ha='center', va='top', fontsize=11, transform=ax.transData)

ax.set_xlim(-1, (steps - 1) * 2.0 + 1)
ax.set_ylim(-0.5, 1.3)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('RNN 展开：每一步复用相同参数', fontsize=12, pad=5)

plt.tight_layout()
out_path = os.path.join(FIGURES_DIR, 'fig_rnn_state_update.png')
fig.savefig(out_path, dpi=600, bbox_inches='tight')
plt.show()
print(f"已保存: {out_path}")



## 批量导出脚本

运行下方代码可一次性查看所有已生成的图片列表。


In [ ]:

import glob
fig_files = sorted(glob.glob(os.path.join(FIGURES_DIR, 'fig_*.png')))
print("当前输出目录中的图片：")
for f in fig_files:
    size_kb = os.path.getsize(f) / 1024
    print(f"  {os.path.basename(f):50s} {size_kb:8.1f} KB")
print(f"\n共 {len(fig_files)} 张图片")
